> ⚠️ **作業中 (Work in Progress)**:このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [Foundry IQ概要](#foundry-iq概要)
- [AI Search接続](#ai-search接続)
- [Knowledge Base作成 (AI Search Index)](#knowledge-base作成-ai-search-index)
- [Knowledge Base作成 (Blob Storage)](#knowledge-base作成-blob-storage)
- [KnowledgeAgent統合](#knowledgeagent統合)

## 🎯 学習目標

- Foundry IQの概念とメリットの理解
- Azure AI Searchリソースの接続と構成
- AI Search IndexベースのKnowledge Base作成
- Blob StorageベースのKnowledge Base
- Knowledge Baseをエージェントに統合する方法の学習

## ⏱️ 予想所要時間

約40分

## Foundry IQ概要

### Foundry IQとは?

Foundry IQは Microsoft Foundryのインテリジェント知識管理システムで, 様々なデータソースを統合して AI エージェントに文脈的知識を提供します.

### 主要特徴

```
Foundry IQ = Retrieval + Reasoning + Ranking
```

- **Retrieval**:関連情報を効率的で検索
- **Reasoning**:検索された情報をが理解し解釈
- **Ranking**:が章関連性高いは情報を優先順位付け

## 環境設定

Knowledge Base 構築をための Azure リソースを設定します.

### Azure 環境変数およびパッケージロード

In [ ]:
# 環境変数ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを見つけられるように)
possible_paths = [
  "/opt/homebrew/bin", # macOS (Apple Silicon)
  "/usr/local/bin",   # macOS (Intel) / Linux
  "/usr/bin",      # Linux / GitHub Codespaces
  "/home/linuxbrew/.linuxbrew/bin" # Linux Homebrew
]

az_path = None
try:
  result = subprocess.run(['which', 'az'], capture_output=True, text=True)
  if result.returncode == 0:
    az_path = os.path.dirname(result.stdout.strip())
except:
  pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
  paths_to_add.append(az_path)
else:
  for path in possible_paths:
    if os.path.exists(path) and path not in os.environ.get("PATH", ""):
      paths_to_add.append(path)

if paths_to_add:
  new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
  os.environ["PATH"] = new_path

# が前ノートブックで保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  # 環境変数設定
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  LOCATION = config.get("LOCATION")
  TENANT_ID = config.get("TENANT_ID")
  PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
  PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
  
  # 環境変数でも設定 (他のツールが使用できるように)
  os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
  os.environ["LOCATION"] = LOCATION
  os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
  os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
  os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
  
  print(f"✅ 設定ファイル '{config_file}'で環境変数をロードしました.")
  print(f"\n📌 Foundry Name:{FOUNDRY_NAME}")
  print(f"📌 Resource Group:{RESOURCE_GROUP}")
  print(f"📌 Location:{LOCATION}")
  print(f"📌 プロジェクトエンドポイント:{PROJECT_ENDPOINT}")
  
except FileNotFoundError:
  print(f"⚠️ '{config_file}' ファイルを見つかりません.")
  print("💡 01-setup.ipynbを先に実行して環境を設定してください.")
  raise

# 必須パッケージインストール
%pip install -q azure-ai-projects azure-identity azure-search-documents requests

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

# Project Client 初期化
credential = DefaultAzureCredential()

# プロジェクト名前抽出 (URLで最後部分)
# はい:https://foundry-xxx.services.ai.azure.com/api/projects/default-project
# → project_name = "default-project"
import re
match = re.search(r'/projects/([^/]+)$', PROJECT_ENDPOINT)
if match:
  project_name = match.group(1)
else:
  project_name = PROJECT_NAME # fallback to config

# Foundry URL (プロジェクト部分除外)
foundry_base_url = PROJECT_ENDPOINT.rsplit('/projects/', 1)[0]

project_client = AIProjectClient(
  endpoint=foundry_base_url,
  credential=credential,
  project_name=project_name
)

print(f"\n💡 使用するプロジェクトエンドポイント:{PROJECT_ENDPOINT}")
print(f"✅ Project Client 初期化完了")
print(f"  Foundry:{foundry_base_url}")
print(f"  Project:{project_name}")

### AI Search および Storage リソース名前作成

In [ ]:
# AI Search および Storage リソース名前設定
import re

# 一意の名前作成 (小文字, 数字, するがオープンだけ許可)
def sanitize_name(name, max_length=24):
  """リソース名前を Azure ルールにに合わせて整理"""
  # 小文字で変換し英数字だけ残す
  clean = re.sub(r'[^a-z0-9]', '', name.lower())
  return clean[:max_length]

# FOUNDRY_NAME ベースで一意名前作成
base_name = sanitize_name(FOUNDRY_NAME)

SEARCH_NAME = f"{base_name}-search"[:64] # AI Searchは最大 64者
STORAGE_NAME = sanitize_name(base_name + "store", 24) # Storageは最大 24者, するがオープン不が
SEARCH_INDEX_NAME = "knowledge-index"

print("📌 作成するリソース名前:")
print(f"  AI Search:{SEARCH_NAME}")
print(f"  Storage Account:{STORAGE_NAME}")
print(f"  Search Index:{SEARCH_INDEX_NAME}")

# 設定ファイルに保存
config["SEARCH_NAME"] = SEARCH_NAME
config["STORAGE_NAME"] = STORAGE_NAME
config["SEARCH_INDEX_NAME"] = SEARCH_INDEX_NAME

with open(config_file, 'w') as f:
  json.dump(config, f, indent=2)

print(f"\n✅ リソース名前が '{config_file}'に保存なりました.")

## Azure AI Search 作成

ベクトル検索をための AI Search リソースを作成します.

### AI Searchリソースの作成実行

In [ ]:
# AI Search リソース作成
!az search service create \
  --name $SEARCH_NAME \
  --resource-group $RESOURCE_GROUP \
  --location $LOCATION \
  --sku basic

print(f"\n✅ AI Search 作成完了:{SEARCH_NAME}")

## Blob Storage 作成

ドキュメントファイルを保存する Blob Storageを作成します.

### Storage Account および Container 作成

In [ ]:
# Storage Account 作成
!az storage account create \
  --name $STORAGE_NAME \
  --resource-group $RESOURCE_GROUP \
  --location $LOCATION \
  --sku Standard_LRS \
  --public-network-access Enabled

print(f"\n✅ Storage Account 作成完了:{STORAGE_NAME}")

# Storage Container 作成
!az storage container create \
  --name documents \
  --account-name $STORAGE_NAME \
  --auth-mode login

print(f"✅ Container 'documents' 作成完了!")

## Managed Identity 有効化

AI Searchが Storageと Foundryにアクセスするできるように Managed Identityを設定します.

In [ ]:
# AI Searchの Managed Identity 有効化
!az search service update \
  --name $SEARCH_NAME \
  --resource-group $RESOURCE_GROUP \
  --identity-type SystemAssigned

print(f"\n✅ AI Search Managed Identity 有効化完了")

## IAM権限の設定

Storage Accountと AI Search, Foundry 間の権限を設定します.

### ユーザー およびリソース情報取得

In [ ]:
# 現在ユーザー 情報取得
import subprocess
import json

# 現在ログによるユーザー Object ID
result = subprocess.run(["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"], 
            capture_output=True, text=True)
USER_OBJECT_ID = result.stdout.strip()
print(f"📌 現在ユーザー Object ID:{USER_OBJECT_ID}")

# AI Searchの Principal ID 取得
result = subprocess.run([
  "az", "search", "service", "show",
  "--name", SEARCH_NAME,
  "--resource-group", RESOURCE_GROUP,
  "--query", "identity.principalId", "-o", "tsv"
], capture_output=True, text=True)
SEARCH_PRINCIPAL_ID = result.stdout.strip()
print(f"📌 AI Search Principal ID:{SEARCH_PRINCIPAL_ID}")

# Storage Account Resource ID 取得
result = subprocess.run([
  "az", "storage", "account", "show",
  "--name", STORAGE_NAME,
  "--resource-group", RESOURCE_GROUP,
  "--query", "id", "-o", "tsv"
], capture_output=True, text=True)
STORAGE_RESOURCE_ID = result.stdout.strip()
print(f"📌 Storage Resource ID:{STORAGE_RESOURCE_ID}")

### Storage Blob Data Contributor ロールの割り当て (ユーザー & AI Search MI)

In [ ]:
# 1. Storage Blob Data Contributor - 現在ユーザー
print("1️⃣ Storage Blob Data Contributor ロールの割り当て (現在ユーザー)...")
!az role assignment create \
  --role "Storage Blob Data Contributor" \
  --assignee $USER_OBJECT_ID \
  --scope $STORAGE_RESOURCE_ID

# 2. Storage Blob Data Contributor - AI Search (Managed Identity)
print("\n2️⃣ Storage Blob Data Contributor ロールの割り当て (AI Search)...")
!az role assignment create \
  --role "Storage Blob Data Contributor" \
  --assignee $SEARCH_PRINCIPAL_ID \
  --scope $STORAGE_RESOURCE_ID

print("\n✅ Storage IAM 権限設定完了!")

### Azure AI Project Manager ロールの割り当て (AI Search MI)

In [ ]:
# 3. Foundry リソースに Azure AI Project Manager ロールの割り当て (AI Search MI)
import subprocess
import os

# Foundry Resource ID 取得
result = subprocess.run([
  "az", "cognitiveservices", "account", "show",
  "--name", FOUNDRY_NAME,
  "--resource-group", RESOURCE_GROUP,
  "--query", "id", "-o", "tsv"
], capture_output=True, text=True)
FOUNDRY_RESOURCE_ID = result.stdout.strip()

print("3️⃣ Azure AI Project Manager ロールの割り当て (AI Search → Foundry)...")
!az role assignment create \
  --role "Azure AI Project Manager" \
  --assignee $SEARCH_PRINCIPAL_ID \
  --scope $FOUNDRY_RESOURCE_ID

print("\n✅ Foundry IAM 権限設定完了!")

## サンプルデータアップロード

Microsoftで提供するはサンプルデータをダウンロードして Storage Containerにアップロードします.

In [ ]:
# サンプルデータダウンロードおよびアップロード
import os
import urllib.request

# サンプルデータ URL (Azure Search Sample Data)
sample_files = [
  ("Benefit_Options.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Benefit_Options.pdf"),
  ("employee_handbook.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/employee_handbook.pdf"),
  ("Northwind_Health_Plus_Benefits_Details.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Northwind_Health_Plus_Benefits_Details.pdf"),
  ("Northwind_Standard_Benefits_Details.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/Northwind_Standard_Benefits_Details.pdf"),
  ("PerksPlus.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/PerksPlus.pdf"),
  ("role_library.pdf", "https://github.com/Azure-Samples/azure-search-sample-data/raw/main/health-plan/role_library.pdf"),
]

# 一時ディレクトリ作成
os.makedirs("temp_data", exist_ok=True)

# ファイルダウンロード
print("📥 サンプルデータダウンロード中...")
for filename, url in sample_files:
  filepath = f"temp_data/{filename}"
  if not os.path.exists(filepath):
    print(f" ダウンロード:{filename}")
    urllib.request.urlretrieve(url, filepath)
  else:
    print(f" が未存在:{filename}")

print("\n✅ サンプルデータダウンロード完了!")

### Storageにサンプルデータアップロード

In [ ]:
# Storage Containerにファイルアップロード
print("📤 Storage Containerにアップロード中...")

for filename, _ in sample_files:
  filepath = f"temp_data/{filename}"
  print(f" アップロード:{filename}")
  !az storage blob upload \
    --account-name $STORAGE_NAME \
    --container-name documents \
    --file $filepath \
    --name $filename \
    --auth-mode login \
    --overwrite

# アップロードされたファイル確認
print("\n📋 アップロードされたファイルリスト:")
!az storage blob list \
  --account-name $STORAGE_NAME \
  --container-name documents \
  --auth-mode login \
  --query "[].name" \
  --output table

print("\n✅ サンプルデータアップロード完了!")

# 一時ファイル整理 (選択事項)
# import shutil
# shutil.rmtree("temp_data")

## AI Search Index 作成 (Portalで進行)

**⚠️ 重要**:次のステップは Azure Portalで手動で進行する必要がありします.

### Import Data Wizard 使用方法:

1. **Azure Portalで AI Search リソース開く**
  - https://portal.azure.com
  - 作成したAI Search サービス選択

2. **Import data (new) クリック**
  
3. **Data Source 設定:**
  - Data Source:**Azure Blob Storage**
  - Scenario:**RAG (Retrieval Augmented Generation)**
  - Storage account:`foundry<your-name>`
  - Container:`documents`

4. **Vectorization 設定:**
  - Kind:**Microsoft Foundry**
  - Foundry project:`proj-default`
  - Model deployment:**text-embedding-3-large**
  - Authentication type:**API key**

5. **Semantic Ranker 有効化:**
  - ☑ Enable semantic ranker
  - Schedule:**Once** (初期インデキシングだけ)

6. **Review + Create**
  - 設定確認後 **Create** クリック
  - インデキシング完了まで 5-10分所要

完了後 以下コードでインデックスを確認してください.

## Python SDKでAI Search Indexを作成

Azure Search Python SDKを使用してベクトル検索が可能なインデックスを作成します.

**2つのがないオプション:**
1. **オプション A (推奨)**:ポータルの Import Data Wizard 使用 - 自動でデータインデキシング
2. **オプション B**:以下 Python コード使用 - インデックスだけ作成 (データは別途アップロード必要)

### Azure Search SDK インストールおよび Import

In [ ]:
# 必要なパッケージのインストール
%pip install -q azure-search-documents azure-identity

from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
  SearchIndex, 
  SearchField, 
  SearchFieldDataType,
  VectorSearch, 
  VectorSearchProfile, 
  HnswAlgorithmConfiguration,
  SemanticConfiguration,
  SemanticField,
  SemanticPrioritizedFields,
  SemanticSearch
)
from azure.identity import DefaultAzureCredential

print("✅ パッケージ import 完了")

### AI Search API Key 獲得

In [ ]:
# API キー 方式で認証 (RBAC 代替案)
# サブスクリプション所有者は API キーを取得するできるあります
result = subprocess.run([
  "az", "search", "admin-key", "show",
  "--resource-group", RESOURCE_GROUP,
  "--service-name", SEARCH_NAME,
  "--query", "primaryKey", "-o", "tsv"
], capture_output=True, text=True)

SEARCH_API_KEY = result.stdout.strip()

if SEARCH_API_KEY:
  print("✅ AI Search API キー 獲得完了")
  print("💡 API キー 方式を使用すると RBAC ロールないがも Indexを作成するできるあります.")
else:
  print("⚠️ API キー 獲得失敗. DefaultAzureCredential 方式を使用します.")

### Search Index 作成 (ベクトル検索 + Semantic Search)

In [ ]:
# AI Search Index 作成
from azure.core.credentials import AzureKeyCredential

# 認証方式選択:API キー まず, なければ DefaultAzureCredential
if 'SEARCH_API_KEY' in globals() and SEARCH_API_KEY:
  credential = AzureKeyCredential(SEARCH_API_KEY)
  auth_method = "API キー"
else:
  credential = DefaultAzureCredential()
  auth_method = "DefaultAzureCredential (RBAC)"

search_endpoint = f"https://{SEARCH_NAME}.search.windows.net"

# SearchIndexClient 作成
index_client = SearchIndexClient(
  endpoint=search_endpoint,
  credential=credential
)

print(f"📌 AI Search Endpoint:{search_endpoint}")
print(f"📌 作成する Index 名前:{SEARCH_INDEX_NAME}")
print(f"🔐 認証方式:{auth_method}")

# Index フィールド定の
fields = [
  # 一意識別子
  SearchField(
    name="chunk_id",
    type=SearchFieldDataType.String,
    key=True,
    sortable=True,
    filterable=True
  ),
  # 親ドキュメント ID
  SearchField(
    name="parent_id",
    type=SearchFieldDataType.String,
    filterable=True
  ),
  # ドキュメントのタイトル
  SearchField(
    name="title",
    type=SearchFieldDataType.String,
    searchable=True,
    filterable=True,
    sortable=True
  ),
  # チャンクされたテキスト内容
  SearchField(
    name="chunk",
    type=SearchFieldDataType.String,
    searchable=True,
    analyzer_name="ko.microsoft" # 韓国語分析期
  ),
  # ベクトルエンベディング (text-embedding-3-large:3072 次元)
  SearchField(
    name="text_vector",
    type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
    searchable=True,
    vector_search_dimensions=3072,
    vector_search_profile_name="my-vector-profile"
  ),
  # メタデータ
  SearchField(
    name="category",
    type=SearchFieldDataType.String,
    filterable=True,
    facetable=True
  ),
  SearchField(
    name="sourcepage",
    type=SearchFieldDataType.String,
    filterable=True
  ),
  SearchField(
    name="sourcefile",
    type=SearchFieldDataType.String,
    filterable=True
  )
]

# ベクトル検索構成
vector_search = VectorSearch(
  profiles=[
    VectorSearchProfile(
      name="my-vector-profile",
      algorithm_configuration_name="my-hnsw-config"
    )
  ],
  algorithms=[
    HnswAlgorithmConfiguration(
      name="my-hnsw-config",
      parameters={
        "m":4, # グラフ接続できる
        "efConstruction":400, # インデキシング時 探索範囲
        "efSearch":500, # 検索時 探索範囲
        "metric":"cosine" # 類似も測定方式
      }
    )
  ]
)

# Semantic Search 構成 (選択事項)
semantic_config = SemanticConfiguration(
  name="my-semantic-config",
  prioritized_fields=SemanticPrioritizedFields(
    title_field=SemanticField(field_name="title"),
    content_fields=[
      SemanticField(field_name="chunk")
    ]
  )
)

semantic_search = SemanticSearch(
  configurations=[semantic_config]
)

# Index 作成
index = SearchIndex(
  name=SEARCH_INDEX_NAME,
  fields=fields,
  vector_search=vector_search,
  semantic_search=semantic_search
)

try:
  # Index 作成または更新
  result = index_client.create_or_update_index(index)
  print(f"\n✅ AI Search Index 作成完了!")
  print(f"  Index 名前:{result.name}")
  print(f"  フィールドできる:{len(result.fields)}")
  print(f"  ベクトル検索:有効化 (3072 次元)")
  print(f"  Semantic Search:有効化")
  
  # 設定ファイルに保存
  config["SEARCH_INDEX_NAME"] = SEARCH_INDEX_NAME
  with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
  
except Exception as e:
  print(f"⚠️ Index 作成失敗:{e}")
  print("\n💡 可能な原因:")
  print("  1. AI Search リソースがまだ準備されていないない (いくつかの分 待機)")
  print("  2. 権限問題:")
  print("   - RBAC 方式:'Search Index Data Contributor' ロール必要")
  print("   - サブスクリプション所有者も Data Plane 権限は別途で必要です")
  print("  3. 上の API キー セルをまず実行すると RBAC ないが可能です")

### 作成された Index 情報確認

In [ ]:
# (選択事項) 作成された Index 確認
try:
  # 作成された Index 取得
  created_index = index_client.get_index(SEARCH_INDEX_NAME)
  
  print("📋 Index 詳細情報:")
  print(f"\nフィールドリスト:")
  for field in created_index.fields:
    field_info = f" - {field.name} ({field.type})"
    if field.key:
      field_info += " [KEY]"
    if field.searchable:
      field_info += " [検索可能]"
    if hasattr(field, 'vector_search_dimensions') and field.vector_search_dimensions:
      field_info += f" [ベクトル:{field.vector_search_dimensions}次元]"
    print(field_info)
  
  print(f"\nベクトル検索プで必:{len(created_index.vector_search.profiles) if created_index.vector_search else 0}個")
  print(f"Semantic 構成:{len(created_index.semantic_search.configurations) if created_index.semantic_search else 0}個")
  
  # Index リスト確認
  print("\n📋 全体 Index リスト:")
  indexes = index_client.list_indexes()
  for idx in indexes:
    print(f" - {idx.name}")
  
except Exception as e:
  print(f"⚠️ Index 情報取得失敗:{e}")

## ドキュメントインデキシング (直接プッシュ方式)

Storageの PDF ドキュメントを読んで直接 Indexにアップロードします.
- PDF パージングおよびテキストの抽出
- テキストチャンキング (2000者単位)
- Foundry APIでエンベディング作成
- AI Search Indexにドキュメントアップロード

### 必須パッケージインストール

In [ ]:
# 必要なパッケージのインストール
%pip install -q pypdf azure-storage-blob openai

from azure.storage.blob import BlobServiceClient
from azure.search.documents import SearchClient
from pypdf import PdfReader
from openai import AzureOpenAI
import io
import hashlib
import base64

print("✅ 必要なパッケージ import 完了")

### Storageで PDF ダウンロードおよびパージング

In [ ]:
# Storageで PDF ファイルダウンロード
from azure.identity import DefaultAzureCredential

# Blob Service Client 作成
storage_credential = DefaultAzureCredential()
blob_service_client = BlobServiceClient(
  account_url=f"https://{STORAGE_NAME}.blob.core.windows.net",
  credential=storage_credential
)

container_client = blob_service_client.get_container_client("documents")

# コンテがあなたのすべての PDF ファイル列挙
print("📥 Storageで PDF ファイルダウンロード中...\n")
pdf_files = []

for blob in container_client.list_blobs():
  if blob.name.endswith('.pdf'):
    print(f" ダウンロード:{blob.name}")
    blob_client = container_client.get_blob_client(blob.name)
    pdf_data = blob_client.download_blob().readall()
    pdf_files.append({
      'name':blob.name,
      'data':pdf_data
    })

print(f"\n✅ 合計 {len(pdf_files)}個ファイルダウンロード完了")

### PDF テキストチャンキングおよびメタデータ作成

In [ ]:
# PDF パージングおよびテキストチャンキング
def chunk_text(text, chunk_size=2000, overlap=200):
  """テキストを指定されたサイズでチャンキング"""
  chunks = []
  start = 0
  text_length = len(text)
  
  while start < text_length:
    end = start + chunk_size
    chunk = text[start:end]
    chunks.append(chunk)
    start = end - overlap # オーバーラップ適用
  
  return chunks

print("📄 PDF パージングおよびテキストチャンキング中...\n")
all_chunks = []

for pdf_file in pdf_files:
  try:
    # PDF 読み取り
    pdf_reader = PdfReader(io.BytesIO(pdf_file['data']))
    
    # 全体テキストの抽出
    full_text = ""
    for page_num, page in enumerate(pdf_reader.pages, 1):
      text = page.extract_text()
      if text:
        full_text += text + "\n"
    
    # テキストチャンキング
    chunks = chunk_text(full_text)
    
    # メタデータと一緒に保存
    for chunk_idx, chunk_content in enumerate(chunks):
      chunk_id = hashlib.md5(f"{pdf_file['name']}_{chunk_idx}".encode()).hexdigest()
      
      all_chunks.append({
        'chunk_id':chunk_id,
        'parent_id':pdf_file['name'],
        'title':pdf_file['name'].replace('.pdf', ''),
        'chunk':chunk_content.strip(),
        'sourcefile':pdf_file['name'],
        'sourcepage':f"page_{chunk_idx + 1}",
        'category':'health-plan'
      })
    
    print(f" ✓ {pdf_file['name']}:{len(chunks)}個チャンク作成")
    
  except Exception as e:
    print(f" ⚠️ {pdf_file['name']} パージング失敗:{e}")

print(f"\n✅ 合計 {len(all_chunks)}個チャンク作成完了")

### Azure OpenAIでエンベディング作成および Indexにアップロード

In [ ]:
# Azure OpenAI REST APIでエンベディング作成
print("🔄 エンベディング作成中...\n")

from azure.identity import DefaultAzureCredential
import requests
import json
import time
import subprocess

# エンベディングモデル設定
embedding_model = "text-embedding-3-large"
api_version = "2024-02-01"

# Azure OpenAI エンドポイント検索
print("🔍 リソースグループで Azure OpenAI アカウント見つけるは中...")

# Azure CLIでリソースグループの Cognitive Services アカウント検索
result = subprocess.run([
  "az", "cognitiveservices", "account", "list",
  "--resource-group", RESOURCE_GROUP,
  "--query", "[?kind=='OpenAI'].{name:name, endpoint:properties.endpoint}",
  "-o", "json"
], capture_output=True, text=True)

openai_accounts = json.loads(result.stdout) if result.stdout else []
print(f"📋 発見された OpenAI アカウント:{len(openai_accounts)}個")

if openai_accounts:
  # 最初の番目 Azure OpenAI アカウント使用
  openai_account = openai_accounts[0]
  openai_name = openai_account['name']
  openai_endpoint = openai_account['endpoint']
  
  print(f"✅ Azure OpenAI アカウント:{openai_name}")
  print(f"🔗 Endpoint:{openai_endpoint}")
  
  # デプロイされたモデル確認
  result = subprocess.run([
    "az", "cognitiveservices", "account", "deployment", "list",
    "--name", openai_name,
    "--resource-group", RESOURCE_GROUP,
    "--query", "[].{name:name, model:properties.model.name}",
    "-o", "json"
  ], capture_output=True, text=True)
  
  deployments = json.loads(result.stdout) if result.stdout else []
  print(f"\n📦 デプロイされたモデル:")
  for dep in deployments:
    print(f"  - {dep['name']}:{dep['model']}")
  
  # embedding モデル検索
  embedding_deployment = None
  for dep in deployments:
    if "embedding" in dep['name'].lower() or "embedding" in dep['model'].lower():
      embedding_deployment = dep['name']
      print(f"\n✅ エンベディングモデルデプロイ発見:{embedding_deployment}")
      break
  
  if not embedding_deployment:
    # デフォルト名前で時も
    embedding_deployment = embedding_model
    print(f"\n💡 デフォルトデプロイ名前使用:{embedding_deployment}")
  
  # REST API URL 構成
  embeddings_url = f"{openai_endpoint.rstrip('/')}/openai/deployments/{embedding_deployment}/embeddings?api-version={api_version}"
  token_scope = "https://cognitiveservices.azure.com/.default"
  
else:
  print("⚠️ Azure OpenAI アカウントを見つかりません.")
  print("💡 Foundry リソース自体を使用してみます...")
  
  # Foundry リソースのエンドポイント取得
  result = subprocess.run([
    "az", "cognitiveservices", "account", "show",
    "--name", FOUNDRY_NAME,
    "--resource-group", RESOURCE_GROUP,
    "--query", "properties.endpoint",
    "-o", "tsv"
  ], capture_output=True, text=True)
  
  foundry_endpoint = result.stdout.strip()
  print(f"🔗 Foundry Endpoint:{foundry_endpoint}")
  
  embeddings_url = f"{foundry_endpoint.rstrip('/')}/openai/deployments/{embedding_model}/embeddings?api-version={api_version}"
  token_scope = "https://cognitiveservices.azure.com/.default"

print(f"\n📍 エンベディング API URL:{embeddings_url}")
print(f"🔐 認証 Scope:{token_scope}")

# Azure 認証トークン取得
credential = DefaultAzureCredential()

# バッチでエンベディング作成 (Rate limit 安定した処理)
batch_size = 30 # Rate limit 考慮して作はバッチ
embedded_chunks = []
max_retries = 5
base_retry_delay = 30 # デフォルト待機時間 30秒

print(f"\n⚡ エンベディング作成開始 (合計 {len(all_chunks)}個, バッチ当 {batch_size}個)")
print(f"💡 安定した処理をために Rate limit 発生時 待機します.\n")

for i in range(0, len(all_chunks), batch_size):
  batch = all_chunks[i:i+batch_size]
  batch_texts = [chunk['chunk'] for chunk in batch]
  batch_num = i // batch_size + 1
  total_batches = (len(all_chunks) + batch_size - 1) // batch_size
  
  success = False
  for retry in range(max_retries):
    try:
      # アクセストークン取得
      token = credential.get_token(token_scope)
      
      # REST APIの呼び出し
      headers = {
        "Content-Type":"application/json",
        "Authorization":f"Bearer {token.token}"
      }
      
      payload = {
        "input":batch_texts,
        "dimensions":3072
      }
      
      response = requests.post(
        embeddings_url,
        headers=headers,
        json=payload,
        timeout=60
      )
      
      if response.status_code == 200:
        result = response.json()
        
        # エンベディングをチャンクに追加
        for j, chunk in enumerate(batch):
          chunk['text_vector'] = result['data'][j]['embedding']
          embedded_chunks.append(chunk)
        
        print(f" ✅ バッチ {batch_num}/{total_batches}:{len(embedded_chunks)}/{len(all_chunks)} チャンク完了")
        success = True
        break
        
      elif response.status_code == 429:
        # Rate limit エラー - 指数バックオフで再試も
        if retry < max_retries - 1:
          wait_time = base_retry_delay * (2 ** retry) # 30, 60, 120, 240秒...
          print(f" ⏳ バッチ {batch_num}/{total_batches}:Rate limit. {wait_time}秒待機後 再試も ({retry+1}/{max_retries})...")
          time.sleep(wait_time)
        else:
          print(f" ⚠️ バッチ {batch_num}/{total_batches}:最大再試も回数秒と - スキップ")
          for chunk in batch:
            chunk['text_vector'] = None
            embedded_chunks.append(chunk)
          success = True # 次のバッチで進行
          break
      else:
        print(f" ⚠️ バッチ {batch_num}/{total_batches} 失敗 (HTTP {response.status_code}):{response.text[:150]}")
        for chunk in batch:
          chunk['text_vector'] = None
          embedded_chunks.append(chunk)
        success = True # 次のバッチで進行
        break
      
    except Exception as e:
      print(f" ⚠️ バッチ {batch_num}/{total_batches} エラー:{str(e)[:100]}")
      if retry < max_retries - 1:
        print(f"   再試も {retry + 1}/{max_retries}...")
        time.sleep(5)
      else:
        for chunk in batch:
          chunk['text_vector'] = None
          embedded_chunks.append(chunk)
        success = True
        break
  
  # バッチ間 1秒待機 (Rate limit 防止)
  if i + batch_size < len(all_chunks) and success:
    time.sleep(1)

print(f"\n✅ エンベディング作成完了:{len(embedded_chunks)}個チャンク")
print(f"📊 ベクトル次元:{len(embedded_chunks[0]['text_vector']) if embedded_chunks and embedded_chunks[0].get('text_vector') else 0}次元")

### Foundry MIに Search Index Data Reader ロールの割り当て

In [ ]:
# Indexにドキュメントアップロード
from azure.core.credentials import AzureKeyCredential

# API キーで SearchClient 作成 (RBAC 代わりに API キー 使用)
upload_credential = AzureKeyCredential(SEARCH_API_KEY) if SEARCH_API_KEY else DefaultAzureCredential()

search_client = SearchClient(
  endpoint=search_endpoint,
  index_name=SEARCH_INDEX_NAME,
  credential=upload_credential
)

print("📤 Indexにドキュメントアップロード中...")
print(f"🔐 認証方式:{'API キー' if SEARCH_API_KEY else 'DefaultAzureCredential'}\n")

# バッチアップロード (最大 1000個ずつ)
batch_size = 30
uploaded_count = 0

for i in range(0, len(embedded_chunks), batch_size):
  batch = embedded_chunks[i:i+batch_size]
  
  # None ベクトル削除 (エンベディング失敗した場合)
  valid_batch = [chunk for chunk in batch if chunk['text_vector'] is not None]
  
  if not valid_batch:
    continue
  
  try:
    # ドキュメントアップロード
    result = search_client.upload_documents(documents=valid_batch)
    
    # 成功したドキュメントできるカウント
    succeeded = sum(1 for r in result if r.succeeded)
    uploaded_count += succeeded
    
    print(f" アップロード:{uploaded_count}/{len([c for c in embedded_chunks if c['text_vector']])} チャンク")
    
  except Exception as e:
    print(f" ⚠️ バッチ {i//batch_size + 1} アップロード失敗:{e}")

print(f"\n✅ インデキシング完了!")
print(f"  合計アップロード:{uploaded_count}個ドキュメント")
print(f"  Index:{SEARCH_INDEX_NAME}")
print(f"\n💡 Index 確認:")
print(f"  https://portal.azure.com → {SEARCH_NAME} → Indexes → {SEARCH_INDEX_NAME}")

### Foundryに AI Search Connection 作成 (REST API)

## Knowledge Baseの作成 (コードベース)

**重要**:Knowledge Base APIはプレビュー 機能で, プレビュー バージョン SDKが必要です.

### Azure Search Documents プレビュー バージョンインストール

In [ ]:
# Knowledge Base API サポートをためのプレビュー バージョンインストール
print("📦 プレビュー SDK インストール中...")
%pip install -q --upgrade azure-search-documents==11.7.0b2

print("\n✅ azure-search-documents 11.7.0b2 インストール完了!")
print("  (Knowledge Base API サポート)")

print("\n⚠️ 重要:カーネル再開始必要!")
print("  1. 上部メニュー:Kernel > Restart Kernel")
print("  2. または以下セル実行して自動再開始")
print("\n💡 カーネル再開始後 次のセルから続行進行してください.")

## Foundry Managed Identityに AI Search 権限付与 (**必須**)

**⚠️ 重要**:API Key Connectionだけでは **Agent 実行が不可能**します!

**権限構造:**
- 🔑 **API Key Connection** → ポータルで Knowledge Base 取得用 (Cell 60)
- 🤖 **Managed Identity 権限** → Agentが Knowledge Base 実行用 (**がセクション**)

**なぜ必要なが?**
- Agent 実行時 Foundryの Managed Identityで Knowledge Base MCP エンドポイントにアクセス
- API Keyは Connection 設定用がであり, Agent ランタイムには使用ならないない
- 403 Forbidden エラー 防止をために必ず設定必要

In [ ]:
# Foundry Managed Identityに AI Search 権限付与 (Agent 実行用 - 必須!)
print("🔐 Foundry Managed Identityに AI Search 権限設定中...")
print("=" * 80)

import json
import uuid
import requests
from azure.identity import DefaultAzureCredential
from azure.mgmt.authorization import AuthorizationManagementClient
from azure.mgmt.authorization.models import RoleAssignmentCreateParameters

try:
  # config ファイルで必要な情報のロード
  config_file = '.foundry_config.json'
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  SEARCH_NAME = config.get("SEARCH_NAME")
  SUBSCRIPTION_ID = config.get("AZURE_SUBSCRIPTION_ID") or config.get("SUBSCRIPTION_ID")
  
  # Foundry リソース ID
  foundry_account_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}"
  
  # AI Search リソース ID
  search_resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.Search/searchServices/{SEARCH_NAME}"
  
  credential = DefaultAzureCredential()
  
  # Foundryの Managed Identity Principal ID 取得
  print(f"\n🔍 Foundry Managed Identity 確認中...")
  
  token = credential.get_token("https://management.azure.com/.default")
  headers = {
    "Authorization":f"Bearer {token.token}",
    "Content-Type":"application/json"
  }
  
  foundry_url = f"https://management.azure.com{foundry_account_id}?api-version=2025-09-01"
  response = requests.get(foundry_url, headers=headers)
  
  if response.status_code != 200:
    raise Exception(f"Foundry 情報取得失敗:{response.status_code}")
  
  foundry_data = response.json()
  principal_id = foundry_data.get("identity", {}).get("principalId")
  
  if not principal_id:
    print(f"  ⚠️ Foundryの Managed Identityが有効化ならないませんでした.")
    print(f"  💡 Managed Identity 有効化中...")
    
    # Managed Identity 有効化
    foundry_data["identity"] = {"type":"SystemAssigned"}
    update_response = requests.patch(foundry_url, headers=headers, json={"identity":{"type":"SystemAssigned"}})
    
    if update_response.status_code in [200, 201]:
      principal_id = update_response.json().get("identity", {}).get("principalId")
      print(f"  ✅ Managed Identity 有効化完了!")
    else:
      raise Exception(f"Managed Identity 有効化失敗:{update_response.text}")
  
  print(f"  ✅ Principal ID:{principal_id}")
  
  # Role Assignment 作成
  auth_client = AuthorizationManagementClient(credential, SUBSCRIPTION_ID)
  
  # 必要なロール定の (Agent 実行用)
  roles_to_assign = [
    {
      "name":"Search Index Data Reader",
      "id":"1407120a-92aa-4202-b7e9-c0e197c71c8f",
      "purpose":"Knowledge Base データ読み取り"
    },
    {
      "name":"Search Service Contributor",
      "id":"7ca78c08-252a-4471-8644-bb5ff32d4ba0",
      "purpose":"Knowledge Base MCP エンドポイントアクセス"
    }
  ]
  
  for role in roles_to_assign:
    print(f"\n🚀 '{role['name']}' 権限確認中...")
    print(f"  目的:{role['purpose']}")
    
    role_definition_id = f"/subscriptions/{SUBSCRIPTION_ID}/providers/Microsoft.Authorization/roleDefinitions/{role['id']}"
    
    # 既存 role assignment 確認
    existing_assignments = list(auth_client.role_assignments.list_for_scope(
      scope=search_resource_id,
      filter=f"principalId eq '{principal_id}'"
    ))
    
    has_role = any(
      role['id'] in assignment.role_definition_id
      for assignment in existing_assignments
    )
    
    if has_role:
      print(f"  ✅ '{role['name']}' 権限がが未存在します.")
    else:
      # 新しい Role Assignment 作成
      role_assignment_name = str(uuid.uuid4())
      role_assignment_params = RoleAssignmentCreateParameters(
        role_definition_id=role_definition_id,
        principal_id=principal_id,
        principal_type="ServicePrincipal"
      )
      
      assignment = auth_client.role_assignments.create(
        scope=search_resource_id,
        role_assignment_name=role_assignment_name,
        parameters=role_assignment_params
      )
      
      print(f"  ✅ '{role['name']}' 権限付与完了!")
      print(f"  Assignment ID:{assignment.name}")
  
  print(f"\n" + "=" * 80)
  print(f"✅ Agent 実行用権限設定完了!")
  print(f"\n⚠️ 重要:権限が伝播されはところ 2-3分所要なります.")
  print(f"💡 2-3分後:")
  print(f"  1. Agent テスト実行可能")
  print(f"  2. 403 Forbidden エラー 解決")
  print(f"  3. Knowledge Base MCP エンドポイント正常アクセス")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ 権限設定失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()


### Azure AI Search 接続追加

Foundry IQで Knowledge Baseを使用するにはまず AI Search リソースを接続する必要があります.

**Foundry IQで AI Search 接続すること:**

1. **Azure AI Foundry Portal 接続**
  - https://ai.azure.com 接続
  - 作成した Foundry プロジェクト選択

2. **Foundry IQ メニューでが同**
  - 左側メニューで **Foundry IQ** クリック
  - "Ground your agent in enterprise knowledge" 画面が表示される

3. **AI Search リソース接続**
  - **Azure AI Search resource** ドロップダウンクリック
  - 作成したAI Search サービス選択
  - **Connect** ボタンクリック

4. **接続完了**
  - AI Search リソースが成功的で接続なると Knowledge basesと Indexes タブが有効化なります

> **💡 Tip**:
> - AI Search リソースがリストになければ "Create new resource" リンクをクリックして新しいで作成するできるあります
> - 接続後にはコードで `project_client.connections.list()`で接続を確認するできるあります

## Knowledge Baseの作成

In [ ]:
# Knowledge Baseをためのライブラリ import
import json
import subprocess
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes.models import (
  KnowledgeBase,
  KnowledgeSourceReference,
  KnowledgeBaseAzureOpenAIModel,
  SearchIndexKnowledgeSource,
  SearchIndexKnowledgeSourceParameters,
  SearchIndexFieldReference,
  AzureOpenAIVectorizerParameters,
  KnowledgeRetrievalOutputMode
)

# config ファイルロード
config_file = '.foundry_config.json'
with open(config_file, 'r') as f:
  config = json.load(f)

SEARCH_NAME = config.get("SEARCH_NAME")

# AI Search API キー 獲得 (コマンドで取得)
result = subprocess.run([
  "az", "search", "admin-key", "show",
  "--resource-group", RESOURCE_GROUP,
  "--service-name", SEARCH_NAME,
  "--query", "primaryKey", "-o", "tsv"
], capture_output=True, text=True)

SEARCH_API_KEY = result.stdout.strip()
search_endpoint = f"https://{SEARCH_NAME}.search.windows.net"

# SearchIndexClient 初期化 (API Key 使用)
index_client = SearchIndexClient(
  endpoint=search_endpoint,
  credential=AzureKeyCredential(SEARCH_API_KEY)
)

print("✅ Knowledge Base ライブラリ import 完了")
print(f"  - KnowledgeBase")
print(f"  - KnowledgeSourceReference")
print(f"  - KnowledgeBaseAzureOpenAIModel")
print(f"  - SearchIndexKnowledgeSource")
print(f"  - SearchIndexKnowledgeSourceParameters")
print(f"  - SearchIndexFieldReference")
print(f"✅ SearchIndexClient 初期化完了:{search_endpoint}")


### Knowledge Source 作成 (Search Index 接続)

In [ ]:
print("🔗 1ステップ:Knowledge Source 作成中...")
KNOWLEDGE_SOURCE_NAME = "knowledge-source-01"

try:
  knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE_NAME,
    description="Foundry IQ Knowledge Source from existing search index",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
      search_index_name=SEARCH_INDEX_NAME,
      # 検索に使用するフィールド指定 (実際インデックスフィールド名前使用)
      source_data_fields=[
        SearchIndexFieldReference(name="chunk"), # テキストコンテンツ
        SearchIndexFieldReference(name="title")  # ドキュメントのタイトル
      ],
      # 検索フィールド (キー フィールド - 実際インデックスのキー フィールド名前使用)
      search_fields=[
        SearchIndexFieldReference(name="chunk_id")
      ]
    )
  )
  
  result_ks = index_client.create_or_update_knowledge_source(knowledge_source)
  print(f"  ✅ Knowledge Source 作成完了:{KNOWLEDGE_SOURCE_NAME}")
except Exception as e:
  print(f"  ⚠️ Knowledge Source 作成失敗:{e}")
  print("  💡 が未存在するは場合無視して続行進行します.")

### Knowledge Baseの作成

In [ ]:
print("📚 2ステップ:Knowledge Base 作成中...")
KNOWLEDGE_BASE_NAME = "foundry-knowledge-base"

# configで必要な変数ロード
FOUNDRY_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
SEARCH_INDEX_NAME = config.get("SEARCH_INDEX_NAME", "knowledge-index")
KNOWLEDGE_SOURCE_NAME = config.get("KNOWLEDGE_SOURCE_NAME", "knowledge-source-01")

# Azure OpenAI パラメータ設定
# answer synthesisをための GPT モデル使用
aoai_params = AzureOpenAIVectorizerParameters(
  resource_url=FOUNDRY_ENDPOINT, # Foundry エンドポイント使用
  deployment_name="gpt-4.1", # GPT デプロイ名前
  model_name="gpt-4.1"
)

try:
  knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Foundry IQ Knowledge Base for HR documents",
    retrieval_instructions="がナレッジベースは HR 関連ドキュメントを含みあります. 従業員福利厚生, 採用ポリシー などにに対する質問に回答してください.",
    answer_instructions="検索されたドキュメントを基半で明確で簡潔な回答を提供してください. ドキュメントで直接引用して信頼性を高いがください.",
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS, # 自動回答合成
    knowledge_sources=[
      KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAME)
    ],
    models=[
      KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)
    ]
  )
  
  result_kb = index_client.create_or_update_knowledge_base(knowledge_base)
  print(f"  ✅ Knowledge Base 作成完了!")
  print(f"\n📋 Knowledge Base 情報:")
  print(f"  名前:{KNOWLEDGE_BASE_NAME}")
  print(f"  Knowledge Source:{KNOWLEDGE_SOURCE_NAME}")
  print(f"  Search Index:{SEARCH_INDEX_NAME}")
  print(f"  Output Mode:Answer Synthesis")
  
  # configに保存
  config["KNOWLEDGE_BASE_NAME"] = KNOWLEDGE_BASE_NAME
  config["KNOWLEDGE_SOURCE_NAME"] = KNOWLEDGE_SOURCE_NAME
  with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
  
  print(f"\n💡 Knowledge Base 確認:")
  print(f"  https://portal.azure.com → {SEARCH_NAME} → Knowledge bases")
  
except Exception as e:
  import traceback
  print(f"  ⚠️ Knowledge Base 作成失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print("\n💡 問題解決:")
  print("  1. カーネルを再開始したはない確認")
  print("  2. azure-search-documents プレビュー バージョン(11.7.0b2)がインストールされているか確認")
  print("  3. Azure OpenAI モデル 'gpt-4.1'がデプロイされてあるか確認")
  print("  4. Search API Key 権限が十分か確認")

### KnowledgeAgentの作成

In [ ]:
# Connectionに API Key 設定 (403 Forbidden 解決)
print("🔑 AI Search Connection API Key 設定中...")
print("=" * 80)

import json
import subprocess
import requests
from azure.identity import DefaultAzureCredential

try:
  # config ファイルで必要な情報のロード
  config_file = '.foundry_config.json'
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  SEARCH_NAME = config.get("SEARCH_NAME")
  RESOURCE_GROUP = config.get("RESOURCE_GROUP")
  SUBSCRIPTION_ID = config.get("AZURE_SUBSCRIPTION_ID") or config.get("SUBSCRIPTION_ID")
  PROJECT_ENDPOINT = config.get("PROJECT_ENDPOINT") or config.get("FOUNDRY_ENDPOINT")
  
  # AI Search API キー 獲得
  print(f"\n🔍 AI Search API Key 取得中...")
  result = subprocess.run([
    "az", "search", "admin-key", "show",
    "--resource-group", RESOURCE_GROUP,
    "--service-name", SEARCH_NAME,
    "--query", "primaryKey", "-o", "tsv"
  ], capture_output=True, text=True)
  
  if result.returncode != 0:
    raise Exception(f"API Key 取得失敗:{result.stderr}")
  
  api_key = result.stdout.strip()
  print(f"  ✅ API Key 獲得完了")
  
  # Project Connection ID 確認
  from azure.ai.projects import AIProjectClient
  credential = DefaultAzureCredential()
  project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
  
  connections = project_client.connections.list()
  search_connections = [c for c in connections if 'search' in c.name.lower()]
  
  if not search_connections:
    raise ValueError("AI Search CoConnectionが見つかりません.")
  
  connection = search_connections[0]
  CONNECTION_NAME = connection.name
  
  print(f"\n🔧 Connection 更新中:{CONNECTION_NAME}")
  
  # Connection 更新 (REST API 使用)
  token = credential.get_token("https://management.azure.com/.default")
  headers = {
    "Authorization":f"Bearer {token.token}",
    "Content-Type":"application/json"
  }
  
  # Project リソース ID 抽出
  project_id = PROJECT_ENDPOINT.split("/projects/")[0].replace("https://", "").replace(".api.azureml.ms", "")
  project_name = PROJECT_ENDPOINT.split("/projects/")[1]
  
  # Foundry アカウント情報確認
  FOUNDRY_NAME = config.get("FOUNDRY_NAME")
  foundry_base_url = f"https://management.azure.com/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.CognitiveServices/accounts/{FOUNDRY_NAME}"
  
  # Connection URL
  connection_url = f"{foundry_base_url}/projects/{project_name}/connections/{CONNECTION_NAME}?api-version=2025-09-01"
  
  # Connection 情報取得
  response = requests.get(connection_url, headers=headers)
  if response.status_code == 200:
    connection_data = response.json()
    
    # API Keyで更新
    connection_data["properties"]["credentials"] = {
      "type":"ApiKey",
      "key":api_key
    }
    
    # Connection 更新
    update_response = requests.put(connection_url, headers=headers, json=connection_data)
    
    if update_response.status_code in [200, 201]:
      print(f"  ✅ Connection API Key 設定完了!")
      print(f"\n💡 が第 Agentが Knowledge Baseにアクセスするできるあります.")
    else:
      print(f"  ⚠️ Connection 更新失敗:{update_response.status_code}")
      print(f"  レスポンス:{update_response.text}")
  else:
    print(f"  ⚠️ Connection 取得失敗:{response.status_code}")
    print(f"  レスポンス:{response.text}")
    
    # 代替案:Portalで手動設定案内
    print(f"\n📝 手動設定方法:")
    print(f"  1. https://ai.azure.com 接続")
    print(f"  2. Build → Connections → {CONNECTION_NAME} 選択")
    print(f"  3. 'Edit' クリック")
    print(f"  4. Authentication セクションで 'API Key' 選択")
    print(f"  5. API Key 入力後 保存")
  
  print(f"\n" + "=" * 80)
  print(f"✅ Connection 設定完了!")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ Connection 設定失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print(f"\n💡 手動設定:")
  print(f"  Portalで Connectionの API Keyを直接設定してください.")


### Connection 作成 (API Key 認証)

Agentが Knowledge Baseにアクセスするには AI Search Connectionが API Keyで認証されてよします.

In [ ]:
print("🤖 3ステップ:KnowledgeAgent 作成中...")
print("=" * 80)

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, MCPTool
from azure.identity import DefaultAzureCredential

try:
  # config ファイルで必要な情報のロード
  config_file = '.foundry_config.json'
  with open(config_file, 'r') as f:
    config = json.load(f)
  
  PROJECT_ENDPOINT = config.get("PROJECT_ENDPOINT") or config.get("FOUNDRY_ENDPOINT")
  SEARCH_NAME = config["SEARCH_NAME"]
  KNOWLEDGE_BASE_NAME = config.get("KNOWLEDGE_BASE_NAME", "foundry-knowledge-base")
  
  # AIProjectClient 作成 (Connection 取得用)
  credential = DefaultAzureCredential()
  project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
  
  # Connection 名前確認 (実際存在するは Connection 使用)
  print(f"\n🔍 プロジェクト Connection リスト確認中...")
  connections = project_client.connections.list()
  search_connections = [c for c in connections if 'search' in c.name.lower()]
  
  if not search_connections:
    raise ValueError(f"❌ AI Search CoConnectionが見つかりません.\n"
            f"💡 Portalで AI Search Connectionをまず作成してください:\n"
            f"  https://ai.azure.com → Build → Connections → + Connection → Azure AI Search")
  
  # 最初の番目 Search Connection 使用
  PROJECT_CONNECTION_NAME = search_connections[0].name
  print(f"  ✅ Connection 発見:{PROJECT_CONNECTION_NAME}")
  
  # MCP エンドポイント URL (api-version 含む, API Keyは Connectionを通じて伝達)
  MCP_ENDPOINT = f"https://{SEARCH_NAME}.search.windows.net/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version=2025-11-01-preview"
  
  print(f"\n📋 Agent 設定:")
  print(f"  Project Endpoint:{PROJECT_ENDPOINT}")
  print(f"  Connection:{PROJECT_CONNECTION_NAME}")
  print(f"  MCP Endpoint:{MCP_ENDPOINT}")
  print(f"\n💡 認証方式:Connectionをを通じた API Key 自動伝達")
  
  # Agent Instructions (Microsoft 推奨テンプレート)
  KNOWLEDGE_INSTRUCTIONS = """You are a helpful assistant that must use the knowledge base to answer all the questions from user. You must never answer from your own knowledge under any circumstances.

Every answer must always provide annotations for using the MCP knowledge base tool and render them as:【message_idx:search_idx†source_name】

If you cannot find the answer in the provided knowledge base you must respond with "I don't know".

韓国語で回答してください."""
  
  # MCP Tool 設定 (Connectionを通じて API Key 伝達)
  print(f"\n🔧 MCP Tool 構成中...")
  print(f"  Server Label:knowledge-base")
  print(f"  Connection ID:{PROJECT_CONNECTION_NAME} (API Key 認証)")
  
  mcp_kb_tool = MCPTool(
    server_label="knowledge-base",
    server_url=MCP_ENDPOINT,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME
  )
  
  print(f"\n🚀 Agent 作成中...")
  
  # Agent 作成
  agent = project_client.agents.create_version(
    agent_name="KnowledgeAgent",
    definition=PromptAgentDefinition(
      model="gpt-5.1",
      instructions=KNOWLEDGE_INSTRUCTIONS,
      tools=[mcp_kb_tool]
    )
  )
  
  print(f"  ✅ KnowledgeAgent 作成完了!")
  print(f"\n📋 Agent 情報:")
  print(f"  名前:{agent.name}")
  print(f"  バージョン:{agent.version}")
  print(f"  モデル:gpt-5.1")
  print(f"  MCP Tool:knowledge_base_retrieve")
  
  # configに保存
  config["AGENT_NAME"] = agent.name
  config["AGENT_VERSION"] = agent.version
  config["PROJECT_CONNECTION_NAME"] = PROJECT_CONNECTION_NAME # 実際 Connection 名前保存
  with open(config_file, 'w') as f:
    json.dump(config, f, indent=2)
  
  print(f"\n💡 Agent 確認:")
  print(f"  https://ai.azure.com → Build → Agents → {agent.name}")
  
except Exception as e:
  import traceback
  print(f"  ⚠️ Agent 作成失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print("\n💡 問題解決:")
  print("  1. Project Connectionがまず作成されているか確認")
  print("  2. Managed Identity 権限が設定されているか確認")
  print("  3. Azure CLI ログの状態確認")
  print("  4. gpt-5.1 モデルがデプロイされてあるか確認")

### Agent テスト

> **⚠️ 実行前の必須作業**:[Azure AI Foundry Portal](https://ai.azure.com)で `foundry-knowledge-base` Knowledge Baseを Projectに接続する必要があります.

In [ ]:
# KnowledgeAgent テスト
print("🧪 4ステップ:KnowledgeAgent テスト開始...")
print("=" * 80)

try:
  # Agent 変数検証
  if 'agent' in locals():
    knowledge_agent = agent
    print(f"✅ Agent 変数使用:{knowledge_agent.name} (version:{knowledge_agent.version})")
  elif 'knowledge_agent' not in locals():
    raise NameError("knowledge_agentが定義されていません. 上の KnowledgeAgent 作成セルをまず実行してください.")
  
  # OpenAI Client 検証
  if 'openai_client' not in locals():
    raise NameError("OpenAI clientがありません. Setup セルをまず実行してください.")
  
  # Test Questions (最初の番目だけテスト)
  test_questions = [
    "PerkPlusがカバーするは項目を教えて"
  ]
  
  for i, question in enumerate(test_questions, 1):
    print(f"\n[質問 {i}]:{question}")
    print("-" * 80)
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    
    # Agentを通じてレスポンス作成
    print(f" レスポンス作成中...", flush=True)
    response = openai_client.responses.create(
      conversation=conversation.id,
      input=question,
      extra_body={"agent":{"name":knowledge_agent.name, "type":"agent_reference"}}
    )
    
    # DEBUG:Response 状態確認
    print(f"\n🔍 Response 状態:{response.status}")
    print(f"🔍 Response ID:{response.id}")
    
    if hasattr(response, 'output') and isinstance(response.output, list):
      print(f"🔍 Output 長さ:{len(response.output)}")
      for idx, item in enumerate(response.output):
        print(f"\n Item {idx}:{item.__class__.__name__}")
        if hasattr(item, 'type'):
          print(f"  type:{item.type}")
        if hasattr(item, 'id'):
          print(f"  id:{item.id}")
        if hasattr(item, 'text'):
          print(f"  text (最初の100文字):{item.text[:100]}")
        if item.__class__.__name__ == 'McpApprovalRequest':
          print(f"  ⚠️ Approval必要!")
          print(f"  name:{item.name if hasattr(item, 'name') else 'N/A'}")
          print(f"  arguments:{item.arguments[:200] if hasattr(item, 'arguments') else 'N/A'}")
    
    # レスポンステキストの抽出
    actual_text = ""
    
    if hasattr(response, 'output') and isinstance(response.output, list):
      for item in response.output:
        if hasattr(item, 'text'):
          actual_text = item.text
          break
    
    print(f"\n[レスポンス]:{actual_text if actual_text else '⚠️ レスポンスを見つかりません'}\n")
  
  print("=" * 80)
  print("✅ KnowledgeAgent テスト完了!")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ テスト失敗:{e}")
  traceback.print_exc()

## Azure Blob Storage ベースのKnowledge Base

Blob Storageを直接接続してより簡単に Knowledge Baseを作成します.

### 1. Storage Account 情報確認

In [ ]:
# config ファイルで Storage Account 名前ロード
try:
  STORAGE_NAME = config.get("STORAGE_NAME")
  if not STORAGE_NAME:
    raise ValueError("STORAGE_NAMEが configにありません.")
except:
  print("⚠️ configで STORAGE_NAMEを見つかりません.")
  print("💡 上の '環境変数ロード' セルと 'AI Search および Storage リソース名前作成' セルをまず実行してください.")
  raise

# Container 名前設定
CONTAINER_NAME = "documents"

print(f"Storage Account:{STORAGE_NAME}")
print(f"Container:{CONTAINER_NAME}")
print(f"\n✅ が未作成された Storage Accountと Containerを使用します.")
print(f"💡 同じ Blob Storageを使用するためで追加設定不必要")

### 2. Knowledge Source 作成 (Blob Storage)

In [ ]:
import subprocess
from azure.search.documents.indexes.models import (
  AzureBlobKnowledgeSource,
  AzureBlobKnowledgeSourceParameters,
  KnowledgeSourceIngestionParameters,
  KnowledgeSourceContentExtractionMode,
  KnowledgeSourceAzureOpenAIVectorizer,
  AzureOpenAIVectorizerParameters
)

# Knowledge Source 名前
BLOB_KNOWLEDGE_SOURCE_NAME = "ks-azureblob-200"

print("Knowledge Source 作成中...")
print(f"名前:{BLOB_KNOWLEDGE_SOURCE_NAME}")

try:
  # Storage Account Resource ID 構成
  storage_resource_id = f"/subscriptions/{SUBSCRIPTION_ID}/resourceGroups/{RESOURCE_GROUP}/providers/Microsoft.Storage/storageAccounts/{STORAGE_NAME}"
  
  # Embedding Model (Vectorizer) 設定
  aoai_vectorizer_params = AzureOpenAIVectorizerParameters(
    resource_url=FOUNDRY_ENDPOINT,
    deployment_name="text-embedding-3-large",
    model_name="text-embedding-3-large"
  )
  
  embedding_vectorizer = KnowledgeSourceAzureOpenAIVectorizer(
    azure_open_ai_parameters=aoai_vectorizer_params
  )
  
  # Ingestion パラメータ設定 (System-Assigned MIは identity パラメータ不必要)
  ingestion_params = KnowledgeSourceIngestionParameters(
    content_extraction_mode=KnowledgeSourceContentExtractionMode.MINIMAL,
    embedding_model=embedding_vectorizer
  )
  
  # Blob Storage パラメータ設定 (Managed Identity Connection String 使用)
  # System-Assigned Managed Identityを使用する時は ResourceId 形式の connection stringだけ必要
  managed_identity_connection_string = f"ResourceId={storage_resource_id};"
  
  blob_params = AzureBlobKnowledgeSourceParameters(
    connection_string=managed_identity_connection_string,
    container_name=CONTAINER_NAME,
    ingestion_parameters=ingestion_params
  )
  
  # Blob Storage ベース Knowledge Source 作成
  blob_knowledge_source = AzureBlobKnowledgeSource(
    name=BLOB_KNOWLEDGE_SOURCE_NAME,
    azure_blob_parameters=blob_params,
    description="Blob Storage ベース Knowledge Source"
  )
  
  # Knowledge Source 作成
  result_blob_ks = index_client.create_knowledge_source(blob_knowledge_source)
  
  print(f"\n✅ Knowledge Source 作成完了!")
  print(f"  - 名前:{result_blob_ks.name}")
  print(f"  - タイプ:Blob Storage")
  print(f"  - Container:{CONTAINER_NAME}")
  print(f"  - Embedding モデル:text-embedding-3-large")
  print(f"\n💡 が Knowledge Sourceは Blob Storageのファイルを自動でインデキシングします.")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ Knowledge Source 作成失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print("\n💡 解決方法:")
  print("  1. Storage Accountと Containerが正しいか確認")
  print("  2. AI Search Managed Identityが Storageにに対する権限があるか確認")
  print("  3. FOUNDRY_ENDPOINTが正しく設定されているか確認")
  print("  4. が未存在するは場合他の名前使用")

### 3. Knowledge Base 作成 (Blob Storage ベース)

> **⚠️ 実行前の必須作業**:[Azure AI Foundry Portal](https://ai.azure.com)で `blob-knowledge-base` Knowledge Baseを Projectに接続する必要があります.

In [ ]:
from azure.search.documents.indexes.models import (
  KnowledgeSourceReference,
  KnowledgeBaseAzureOpenAIModel
)

# Knowledge Base 名前
BLOB_KNOWLEDGE_BASE_NAME = "knowledgebase200"

print("Knowledge Base 作成中...")
print(f"名前:{BLOB_KNOWLEDGE_BASE_NAME}")

try:
  # Azure OpenAI モデル設定 (Answer Synthesis用)
  aoai_model_params = AzureOpenAIVectorizerParameters(
    resource_url=FOUNDRY_ENDPOINT,
    deployment_name="gpt-5.1",
    model_name="gpt-5.1"
  )
  
  # Knowledge Base 作成 (Blob Storage ベース)
  blob_knowledge_base = KnowledgeBase(
    name=BLOB_KNOWLEDGE_BASE_NAME,
    description="Blob Storage ベースのKnowledge Base",
    retrieval_instructions="がナレッジベースは Blob Storageのドキュメントを含みあります.",
    answer_instructions="検索されたドキュメントを基半で明確な回答を提供してください.",
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    knowledge_sources=[
      KnowledgeSourceReference(name=BLOB_KNOWLEDGE_SOURCE_NAME)
    ],
    models=[
      KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_model_params)
    ]
  )
  
  # Knowledge Base 作成
  result_blob_kb = index_client.create_or_update_knowledge_base(blob_knowledge_base)
  
  print(f"\n✅ Knowledge Base 作成完了!")
  print(f"  - 名前:{result_blob_kb.name}")
  print(f"  - Output Mode:{result_blob_kb.output_mode}")
  print(f"  - Knowledge Source:{BLOB_KNOWLEDGE_SOURCE_NAME}")
  print(f"  - モデル:gpt-5.1")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ Knowledge Base 作成失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print("\n💡 解決方法:")
  print("  1. Knowledge Sourceがまず作成されているか確認")
  print("  2. が未存在するは場合他の名前使用")
  print("  3. Chat モデル名前確認")

### 4. KnowledgeAgent2 作成 (Blob Storage ベース)

In [ ]:
KNOWLEDGE_AGENT_2_NAME = "KnowledgeAgent2"

print("KnowledgeAgent2 作成中...")
print(f"Knowledge Base:{BLOB_KNOWLEDGE_BASE_NAME}")

try:
  # MCP Tool 定の (Blob Storage ベースのKnowledge Base)
  mcp_blob_kb_tool = MCPTool(
    server_label="knowledge-base-blob",
    server_url=MCP_ENDPOINT,
    require_approval="never",
    allowed_tools=["knowledge_base_retrieve"],
    project_connection_id=PROJECT_CONNECTION_NAME
  )
  
  # Agent 作成
  blob_knowledge_agent = project_client.agents.create_version(
    agent_name=KNOWLEDGE_AGENT_2_NAME,
    definition=PromptAgentDefinition(
      model="model-router",
      instructions=KNOWLEDGE_INSTRUCTIONS,
      tools=[mcp_blob_kb_tool]
    )
  )
  
  print(f"\n✅ {KNOWLEDGE_AGENT_2_NAME} 作成完了!")
  print(f"  - Agent:{KNOWLEDGE_AGENT_2_NAME}")
  print(f"  - Model:model-router")
  print(f"  - Knowledge Base:{BLOB_KNOWLEDGE_BASE_NAME}")
  print(f"  - データソース:Blob Storage")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ Agent 作成失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()
  print("\n💡 解決方法:")
  print("  1. Knowledge Baseがまず作成されているか確認")
  print("  2. MCP endpoint 確認")
  print("  3. PROJECT_CONNECTION_NAMEが正しいか確認")

### 5. KnowledgeAgent2 テスト

In [ ]:
import time
import json

print(f"{'=' * 80}")
print(f"KnowledgeAgent2 テスト (Blob Storage ベース)")
print(f"{'=' * 80}\n")

try:
  # OpenAI client 取得
  openai_client = project_client.get_openai_client()
  
  for i, question in enumerate(test_questions, 1):
    print(f"\n[質問 {i}]:{question}")
    print("-" * 80)
    
    # Conversation 作成
    conversation = openai_client.conversations.create()
    
    # Agentを通じてレスポンス作成 (Rate limit 再試もで直含む)
    print(f" レスポンス作成中...", flush=True)
    
    max_retries = 3
    retry_delay = 15 # 秒
    response = None
    
    for attempt in range(max_retries):
      try:
        response = openai_client.responses.create(
          conversation=conversation.id,
          input=question,
          extra_body={"agent":{"name":blob_knowledge_agent.name, "type":"agent_reference"}}
        )
        print(f" ✅ レスポンス作成成功!")
        break # 成功するとループ終了
        
      except Exception as e:
        error_message = str(e)
        if "429" in error_message or "Too Many Requests" in error_message:
          if attempt < max_retries - 1:
            wait_time = retry_delay * (attempt + 1)
            print(f" ⏳ Rate limit も月. {wait_time}秒待機後 再試も... ({attempt + 1}/{max_retries})")
            time.sleep(wait_time)
          else:
            print(f" ❌ {max_retries}回再試も後にも失敗.")
            print(f" 💡 5分定も待った後 再度時もしてください.")
            raise
        else:
          # 他のエラーはもので throw
          raise
    
    if response is None:
      print("⚠️ レスポンスを受けないできませんでした.")
      continue
    
    # DEBUG:Response 状態確認
    print(f"\n🔍 Response 状態:{response.status}")
    print(f"🔍 Response ID:{response.id}")
    
    if hasattr(response, 'output') and isinstance(response.output, list):
      print(f"🔍 Output 長さ:{len(response.output)}")
      for idx, item in enumerate(response.output):
        print(f"\n Item {idx}:{item.__class__.__name__}")
        if hasattr(item, 'type'):
          print(f"  type:{item.type}")
        if hasattr(item, 'id'):
          print(f"  id:{item.id}")
        if hasattr(item, 'text'):
          print(f"  text (最初の100文字):{item.text[:100]}")
        if item.__class__.__name__ == 'McpApprovalRequest':
          print(f"  ⚠️ Approval必要!")
          print(f"  name:{item.name if hasattr(item, 'name') else 'N/A'}")
          print(f"  arguments:{item.arguments[:200] if hasattr(item, 'arguments') else 'N/A'}")
    
    # レスポンステキストの抽出
    actual_text = ""
    
    if hasattr(response, 'output') and isinstance(response.output, list):
      for item in response.output:
        if hasattr(item, 'text'):
          actual_text = item.text
          break
    
    print(f"\n[レスポンス]:{actual_text if actual_text else '⚠️ レスポンスを見つかりません'}\n")
  
  print("=" * 80)
  print("✅ KnowledgeAgent2 テスト完了!")
  
except Exception as e:
  import traceback
  print(f"\n⚠️ テスト失敗:{e}")
  print("\n詳細エラー:")
  traceback.print_exc()


### 📝 2つの方式比較

| 特徴 | AI Search Index | Blob Storage 直接接続 |
|------|----------------|----------------------|
| **設定複雑も** | 高い (Import Wizard 必要) | 低い (簡単な設定) |
| **自動更新** | 手動再インデキシング必要 | 自動検出およびインデキシング |
| **カスタマが徴** | 高い (フィールド, スキーマなど) | 低い (自動構成) |
| **パフォーマンス** | 高い (最適化可能) | 中間 |
| **使用事例** | 複雑な検索要件 | 簡単なドキュメント検索 |

**推奨事項:**
- 複雑な検索およびフィルターリングが必要な場合 → AI Search Index
- 迅速なプでプロトタイプおよび簡単なドキュメント検索 → Blob Storage

## 📚 追加リソース

- [Foundry IQ 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/knowledge-retrieval?view=foundry&tabs=foundry%2Cpython)
- [Azure AI Search ドキュメント](https://learn.microsoft.com/en-us/azure/search/)
- [RAG パターンガイド](https://learn.microsoft.com/en-us/azure/search/retrieval-augmented-generation-overview?tabs=docs)
- [ベクトル検索最適化](https://learn.microsoft.com/en-us/azure/search/vector-search-overview)